# 🏁 归因分析马拉松：0-1 构建多触点归因模型 (Multi-Touch Attribution)

> **本次马拉松目标**：
> 解决营销界最头疼的问题：用户看了 3 个广告才购买，功劳怎么分？
>
> **核心技能点**：
> 1.  **路径数据处理**: 处理 User Journey (例如: "Search -> Social -> Email -> Conversion")。
> 2.  **规则归因 (Rule-based)**: 实现 Last-Touch, First-Touch, Linear 归因。
> 3.  **马尔可夫链 (Markov Chain)**: 计算渠道间的 "移除效应" (Removal Effect)，更科学地分配功劳。
> 4.  **ROI 决策**: 结合渠道成本，产出预算调整建议。

---

## 1. 模拟用户路径数据 (Data Generation)
我们需要生成这种格式的数据：
| UserID | Path | Conversion |
| :--- | :--- | :--- |
| U001 | Search > Social > Email | 1 |
| U002 | Search > Social | 0 |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

channels = ['Organic', 'Social_Ads', 'Search_Ads', 'Email', 'Video_Ads']

# 生成 2000 条用户路径
paths = []
conversions = []

for _ in range(2000):
    # 随机生成路径长度 (1~5 步)
    length = np.random.randint(1, 6)
    path = np.random.choice(channels, length, replace=True).tolist()
    
    # 定义转化逻辑：
    # 1. 并没有绝对的逻辑，随机性很大
    # 2. 但是 'Search_Ads' -> 'Email' 这个组合转化率很高
    prob = 0.05
    if 'Email' in path: prob += 0.1
    if 'Search_Ads' in path and 'Email' in path: prob += 0.2
    if 'Video_Ads' in path: prob += 0.05
    
    # 限制概率在 0.8 以内
    prob = min(prob, 0.8)
    
    conv = np.random.binomial(1, prob)
    
    paths.append(' > '.join(path))
    conversions.append(conv)

df = pd.DataFrame({'path': paths, 'conversion': conversions})
print(f"Data Generated. Conversion Rate: {df['conversion'].mean():.2%}")
df.head()

## 2. 规则归因实战 (Rule-based Attribution)
**任务**：
1.  **Last Touch**: 功劳全归最后一个渠道。
2.  **Linear**: 路径上所有渠道平分功劳。

你需要计算出每个渠道带来的**总转化数**。

In [ ]:
# 🔍 你的代码：计算 Last Touch Attribution
# 提示：只看 conversion == 1 的行，取 path 的最后一个元素

# 🔍 你的代码：计算 Linear Attribution
# 提示：对于 conversion == 1 的行，split path，每个渠道得分为 1/len(path)


## 3. 马尔可夫链归因 (Markov Chain Attribution)
这是数据科学对归因领域的最大贡献之一。
**核心思想**：把用户旅程看作图的跳转。计算 "移除某个节点后，整体转化率会下降多少？" (Removal Effect)。

**步骤** (不要被吓到，我们手动实现简化版)：
1.  **构建转移矩阵**: 计算从 A -> B 的概率。
    *   State 需要增加 [Start], [Conversion], [Null] (未转化)。
2.  **计算 Removal Effect**:
    *   移除 'Facebook' 节点，重新计算 [Start] 到 [Conversion] 的概率。
    *   下降的比例，就是 Facebook 的重要性。

*(这部分代码较难，我提供了一个框架，你需要填空)*

In [ ]:
# 🧠 思考题：如何把 'Search > Social' 变成对 (From, To) 的统计？
# Path: Start -> Search -> Social -> Null (未转化)
# Path: Start -> Search -> Social -> Email -> Conversion (转化)

# 你的任务：提取所有的 (Current_State, Next_State) 对，并计算频次


## 4. ROI 预算决策 (Budget Allocation)
单纯看转化数没用，要看 **ROI** (投入产出比)。

假设成本如下：
*   Video_Ads: $50 / conv (很贵)
*   Search_Ads: $20 / conv
*   Social_Ads: $10 / conv
*   Email: $1 / conv

**任务**：
结合你计算出的 Attribution 结果 (Channel Conversions)，计算各渠道的 CPA (Cost Per Action)。
假如老板要砍预算，你建议砍掉哪个？投入哪个？

In [ ]:
# 💰 你的代码：计算各渠道 CPA
costs = {
    'Video_Ads': 5000, # 假设总花费
    'Search_Ads': 2000,
    'Social_Ads': 1000,
    'Email': 100,
    'Organic': 0
}

# CPA = Cost / Attributed_Conversions
